# 🎵 DeepRaaga Tutorial

**Learning and Generating Carnatic Music with AI**

This notebook demonstrates how to use the DeepRaaga PyPI packages for:
1. Data preprocessing from MIDI files
2. Training an LSTM+Attention model
3. Generating new Raga sequences
4. Using the REST API

---

## 📦 Installation

In [ ]:
# Install required packages
!pip install deepraaga-core deepraaga-preprocess deepraaga-models deepraaga-api

---

## 1️⃣ Understanding the Architecture

DeepRaaga is organized into 4 modular packages:

| Package | Purpose | Key Classes/Functions |
|---------|---------|----------------------|
| `deepraaga-core` | Base abstractions | `BaseModel`, `VGGModel` |
| `deepraaga-preprocess` | Data ingestion | `DataProcessor`, `preprocess_ragas()` |
| `deepraaga-models` | Neural networks | `DeepRagaModel`, `train_model()` |
| `deepraaga-api` | REST API | `app`, `load_model()` |

---

## 2️⃣ Data Preprocessing

The `DataProcessor` handles MIDI file parsing, vocabulary building, and sequence creation.

### 2.1 Basic Usage

In [ ]:
from deepraaga_preprocess.data_processor import DataProcessor
import os

# Initialize the processor
processor = DataProcessor(
    sample_rate=22050,      # Audio sample rate
    hop_length=512,         # Hop length for feature extraction
    sequence_length=100     # Sequence length for training
)

print(f"Initialized DataProcessor")
print(f"  Sequence length: {processor.sequence_length}")
print(f"  Sample rate: {processor.sample_rate}")

### 2.2 Process a Single MIDI File

In [ ]:
# Example: Process a single MIDI file
# Note: Replace with actual MIDI file path
midi_path = "path/to/your/raga.mid"

if os.path.exists(midi_path):
    inputs, outputs = processor.extract_midi_features(midi_path, training=True)
    print(f"Extracted {len(inputs)} sequences from {midi_path}")
    print(f"Vocabulary size: {len(processor.note_to_int)}")
    print(f"\nSample notes: {list(processor.note_to_int.keys())[:10]}")
else:
    print(f"MIDI file not found: {midi_path}")
    print("Skipping single file processing demo...")

### 2.3 Process Entire Dataset

In [ ]:
# Process entire dataset
midi_dir = "data/raw"       # Directory with MIDI files
output_dir = "data/processed"

if os.path.exists(midi_dir):
    print(f"Processing dataset from {midi_dir}...")
    processor.process_dataset(midi_dir, output_dir)
    
    # Save vocabulary
    vocab_path = os.path.join(output_dir, "vocab.pkl")
    processor.save_vocab(vocab_path)
    print(f"\nVocabulary saved to {vocab_path}")
else:
    print(f"Dataset directory not found: {midi_dir}")
    print("Skipping dataset processing demo...")

### 2.4 Load Preprocessed Data

In [ ]:
import numpy as np

# Load preprocessed data
X_path = "data/processed/X.npy"
y_path = "data/processed/y.npy"
vocab_path = "data/processed/vocab.pkl"

if os.path.exists(X_path) and os.path.exists(y_path):
    X = np.load(X_path)
    y = np.load(y_path)
    
    print(f"Loaded dataset:")
    print(f"  X shape: {X.shape}  (sequences)")
    print(f"  y shape: {y.shape}  (targets)")
    
    # Load vocabulary
    processor.load_vocab(vocab_path)
    vocab_size = len(processor.note_to_int)
    print(f"  Vocabulary size: {vocab_size}")
else:
    print("Preprocessed data not found. Run dataset processing first.")

---

## 3️⃣ Model Training

### 3.1 Create the DeepRaga Model

In [ ]:
import torch
from deepraaga_models.model import DeepRagaModel

# Model hyperparameters
vocab_size = len(processor.note_to_int) if hasattr(processor, 'note_to_int') else 100
embedding_dim = 64
hidden_size = 256
num_layers = 2
dropout = 0.3

# Initialize model
model = DeepRagaModel(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size,
    num_layers=num_layers,
    dropout=dropout
)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"DeepRaga Model initialized:")
print(f"  Vocab size: {vocab_size}")
print(f"  Embedding dim: {embedding_dim}")
print(f"  Hidden size: {hidden_size}")
print(f"  Num layers: {num_layers}")
print(f"  Device: {device}")
print(f"\nModel architecture:")
print(model)

### 3.2 Create Dataset and DataLoader

In [ ]:
from torch.utils.data import DataLoader, Dataset

class RagaDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.LongTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return {
            'sequence': self.X[idx],
            'target': self.y[idx]
        }

# Create datasets (use dummy data if real data not available)
if os.path.exists(X_path):
    # Load real data
    split_idx = int(0.8 * len(X))
    train_dataset = RagaDataset(X[:split_idx], y[:split_idx])
    val_dataset = RagaDataset(X[split_idx:], y[split_idx:])
else:
    # Create dummy data for demonstration
    dummy_X = torch.randint(0, vocab_size, (1000, 100))
    dummy_y = torch.randint(0, vocab_size, (1000,))
    train_dataset = RagaDataset(dummy_X[:800].numpy(), dummy_y[:800].numpy())
    val_dataset = RagaDataset(dummy_X[800:].numpy(), dummy_y[800:].numpy())
    print("Using dummy data for demonstration")

# Create dataloaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

print(f"\nDatasets created:")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")
print(f"  Batch size: {batch_size}")

### 3.3 Training Loop

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training function
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for batch in loader:
        sequences = batch['sequence'].to(device)
        targets = batch['target'].to(device)
        
        optimizer.zero_grad()
        outputs, _ = model(sequences)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

# Validation function
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in loader:
            sequences = batch['sequence'].to(device)
            targets = batch['target'].to(device)
            
            outputs, _ = model(sequences)
            loss = criterion(outputs, targets)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    accuracy = 100. * correct / total
    return total_loss / len(loader), accuracy

print("Training functions defined.")

### 3.4 Run Training (Demo with 2 epochs)

In [ ]:
# Train for a few epochs (demo)
num_epochs = 2

for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

print("\nTraining demo complete!")

### 3.5 Save the Model

In [ ]:
# Save model
model_dir = "model"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "trained_model.pth")

torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

---

## 4️⃣ Generation

### 4.1 Generate New Raga Sequences

In [ ]:
import numpy as np

def generate_sequence(model, processor, start_note=None, length=50, temperature=1.0, device='cpu'):
    """
    Generate a new note sequence using the trained model.
    
    Args:
        model: Trained DeepRagaModel
        processor: DataProcessor with loaded vocabulary
        start_note: Starting note (random if None)
        length: Number of notes to generate
        temperature: Sampling temperature (higher = more random)
        device: torch device
    
    Returns:
        List of note names
    """
    model.eval()
    
    # Start with random note or specified note
    if start_note is None:
        start_idx = np.random.randint(0, len(processor.note_to_int))
    else:
        start_idx = processor.note_to_int.get(start_note, 0)
    
    generated = [start_idx]
    input_seq = torch.LongTensor([[start_idx]]).to(device)
    hidden = None
    
    with torch.no_grad():
        for _ in range(length):
            output, hidden = model(input_seq, hidden)
            
            # Apply temperature
            logits = output / temperature
            probs = torch.softmax(logits, dim=1)
            
            # Sample next note
            next_idx = torch.multinomial(probs, 1).item()
            generated.append(next_idx)
            
            # Prepare input for next step
            input_seq = torch.LongTensor([[next_idx]]).to(device)
    
    # Convert to note names
    notes = [processor.int_to_note.get(idx, 'C4') for idx in generated]
    return notes

# Generate a sequence
if hasattr(processor, 'note_to_int') and len(processor.note_to_int) > 0:
    generated_notes = generate_sequence(
        model=model,
        processor=processor,
        length=30,
        temperature=0.8,
        device=device
    )
    
    print("Generated Raga Sequence:")
    print(" ".join(generated_notes[:20]) + " ...")
else:
    print("No vocabulary loaded. Cannot generate.")

---

## 5️⃣ Using the REST API

### 5.1 Start the API Server

In [ ]:
# You can also start the API server from command line:
# !deepraaga-api --port 8000

# Or run it programmatically:
from deepraaga_api.serve import app, load_model

# Load model if available
if os.path.exists("model/trained_model.pth"):
    load_model()
    print("Model loaded into API server")
else:
    print("No trained model found. API will run in fallback mode.")

print("\nTo start the server programmatically:")
print("  app.run(host='0.0.0.0', port=8000)")

### 5.2 Test API Endpoints

In [ ]:
import requests
import json

# Test the API (if server is running)
api_url = "http://localhost:8000"

try:
    # Health check
    response = requests.get(f"{api_url}/", timeout=2)
    print(f"API Status: {response.status_code}")
    print(f"Response: {response.json()}")
    
    # Generate sequence
    response = requests.post(
        f"{api_url}/api/generate",
        json={"raga": "Bhairavi", "duration": 30, "temperature": 0.8},
        timeout=5
    )
    print(f"\nGeneration Response: {response.status_code}")
    data = response.json()
    print(f"Notes: {data.get('notes', [])[:10]} ...")
    
except requests.exceptions.ConnectionError:
    print("API server not running. Start it with:")
    print("  deepraaga-api --port 8000")
except Exception as e:
    print(f"Error: {e}")

---

## 6️⃣ Raga Preprocessing Utilities

### 6.1 Convert Swara Patterns to MIDI

In [ ]:
from deepraaga_preprocess.preprocess_raga import swara_to_midi, convert_pattern_to_midi

# Convert individual swaras
swaras = ['S', 'R2', 'G3', 'M1', 'P', 'D2', 'N3', "S'"]
midi_notes = [swara_to_midi(s) for s in swaras]

print("Swara to MIDI conversion:")
for s, m in zip(swaras, midi_notes):
    print(f"  {s:4} -> {m}")

# Convert a pattern
pattern = "S R2 G3 M1 P D2 N3 S'"
midi_pattern = convert_pattern_to_midi(pattern)
print(f"\nPattern: {pattern}")
print(f"MIDI: {midi_pattern}")

---

## 📝 Summary

This notebook demonstrated:

1. **Installation** - Installing DeepRaaga packages from PyPI
2. **Data Preprocessing** - Using `DataProcessor` to process MIDI files
3. **Model Training** - Training an LSTM+Attention model
4. **Generation** - Creating new Raga sequences
5. **API Usage** - Starting and testing the REST API
6. **Utilities** - Swara-to-MIDI conversion

---

## 🔗 Resources

- **Repository**: https://github.com/sgmoorthy/naada
- **PyPI Packages**: https://pypi.org/search/?q=deepraaga
- **Documentation**: See README.md in repository

Happy Music Generation! 🎵